# 中间件

解决需求：
- 想根据问题复杂度动态切换模型 ；
- 想限制某些用户只能调用部分工具；
- 想在工具报错时自动重试或返回兜底结果；
- 想在模型调用前插入额外的系统提示 ；
- 想记录每一步的执行日志 ，方便排查问题；
- 想在敏感信息出现时阻断执行 ；
- 想在正式执行工具前增加人工审批 。
### 中间件的价值就在于把这些与业务无关、但与执行过程强相关的横切逻辑，从 Agent 主流程中分离出来。让Agent 主体代码 聚焦业务 ，而借助中间件，实现“ 拦截流程、修改流程、增强流程 ”。

### 中间件的分类 ：
    自定义中间件，langchain内置中间件

![image.png](images/middleware.png)

对上图进行分类

##### 类型1：成本与资源控制类
核心目标：控成本、控配额、避免无限调用
这类中间件主要解决“ Agent太贵、太能跑、停不下来 ”的问题。
包含：
- Model call limit：限制模型调用次数，防止一次任务反复请求 LLM，导致费用失控
- Tool call limit：限制工具调用次数，避免 Agent 无限试错、死循环调工具
- Summarization：在上下文快满时自动总结历史，减少 token 消耗
- Context editing：裁剪上下文、清理工具调用痕迹，本质上也是为了节省上下文成本

##### 类型2：稳定性与容错保障类
核心目标：保证服务不中断、失败后尽量自动恢复
这类中间件主要解决“ 调用失败怎么办、模型挂了怎么办、工具超时怎么办 ”。
包含：
- Model fallback：主模型失败时切换备用模型
- Model retry：模型调用失败后自动重试
- Tool retry：工具调用失败后自动重试

业务场景理解：
适合线上生产系统，尤其是多模型、多工具依赖的 Agent。

##### 类型3：安全与合规风控类
核心目标：让 Agent 可控、可审、合规
这类中间件主要解决“ Agent乱执行、泄露敏感信息、做危险操作 ”的问题。
包含：
- Human-in-the-loop：在关键工具调用前暂停，等人工审批
- PII detection：检测和处理个人敏感信息
- Model call limit / Tool call limit：某种意义上也可归到风控，因为它能防止异常滥用
业务场景理解：
适合企业内部系统、客服系统、审批流、数据查询类 Agent

##### 类型4：决策增强与智能编排类
核心目标：提升 Agent 的决策质量和任务拆解能力
这类中间件主要解决“ Agent不够聪明、不会规划、不会先筛工具 ”的问题。
包含：
- To-do list：给 Agent 增加任务规划、分步骤执行和状态跟踪能力
- LLM tool selector：当工具太多时，用子模型筛选最相关的几个工具交给主模型
- Subagent：允许生成子Agent，把复杂任务拆给不同角色处理
业务场景理解：
适合复杂任务流，比如：研究型 Agent、多步骤分析、报告生成、多角色协作、长链路任务编排等。

##### 类型5：执行能力扩展类
核心目标：给 Agent 更多“手脚”
这类中间件主要解决“ Agent只能聊天，不能真正操作环境 ”的问题。
包含：
- Shell tool：给 Agent 持久 shell，会执行命令
- File search：给 Agent 文件搜索能力，能做 Glob/Grep
- Filesystem：给 Agent 文件系统读写与长期存储能力
业务场景理解：
适合工程 Agent、代码 Agent、本地自动化 Agent、运维 Agent。
本质上是把 Agent 从“纯推理”扩展成“能操作环境的执行体”。

##### 类型6：开发调试与测试辅助类
核心目标：方便开发、测试、验证 Agent 行为
这类中间件主要不是直接服务业务，而是服务于 研发和调试阶段 。
包含：
- LLM tool emulator：用 LLM 模拟工具执行，便于测试（最典型）
- Summarization：有时也可辅助调试长会话表现
- Context editing：可用于测试上下文裁剪效果
- Human-in-the-loop：也常用于调试高风险步骤
业务场景理解：
适合开发阶段快速验证流程、做 mock、减少真实工具依赖。

### SummarizationMiddleware中间件

作用： 对历史消息列表进行摘要&总结 ，达到压缩上下文的效果。

原理：在达到触发条件时，调用大模型对历史消息进行摘要， 将摘要的结果作为HumanMessage ，放到消息列表最开始的位置。

参数：
1. model
2. trigger（摘要触发条件）
    是一个列表，每个元素对应一个条件，当 任一条件满足 时，触发摘要。
    1. tokens ：token的数量，历史token的累计数量达到该值触发摘要。
    2. messages ：历史消息数量，历史消息条数达到该值触发摘要。
    3. fraction ：上下文长度比例。历史token的累计数量达到模型的 max_input_tokens*fraction 触发摘要

        如果条件包含 fraction ，要求模型的profile包含 max_input_tokens ，Deepseek模型的profile为空，此时需要手动添加该配置项。Deepseek-V3.2的上下文长度为128K。
3. keep （摘要保留的原始信息）
    支持三种条件，但和trigger不同，keep同一时间只接收一种条件
    1. tokens ：摘要时保留的token数量。
    2. messages ：摘要时保留的历史消息条数。
    3. fraction ：摘要时保留 max_input_tokens*fraction 个token。
4. token_counter —统计token数量的函数
5. summary_prompt —摘要时的自定义提示词、
6. trim_token_to_summarize —摘要时历史消息的最大token数 默认为4000

In [2]:
from typing import override

from langchain.chat_models import  init_chat_model
from langchain.agents import  create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from dotenv import load_dotenv
import os

from langchain_classic.memory import summary
from openai import max_retries
from openai.types.shared_params import reasoning_effort

load_dotenv(override=True)
custom_profile={
    "max_input_tokens":128_000
}
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek",
                      profile=custom_profile)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent=create_agent(model="deepseek-flash",
                   middleware=[
                       SummarizationMiddleware(
                           model=model,
                           trigger=[
                               ("tokens",100),
                               ("messages",6),
                               ("fraction",0.001)
                           ],
                           keep=("messages",2)
                       )
                   ])

response=agent.invoke({"messages":messages})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户（老王）本次会话仅进行了自我介绍和打招呼，没有提出具体任务或目标。

## SUMMARY
- 用户自称“老王”。
- AI助手自称“小王”。
- 老王表示很高兴认识小王。
- 本次对话为友好寒暄，尚未确定后续任务或工作方向。

## ARTIFACTS
None

## NEXT STEPS
None。若用户继续对话，可保持友好语气并等待其提出具体需求。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

老王别误会，我那句是接你“很高兴认识小王”的梗，故意逗一下、卖个关子，没有别的意思，更不是说你高兴得不对。要是听着不舒服，我收回——改成：**我也很高兴认识老王。**

有啥需要尽管说，小王在。


In [2]:
##测试summary_prompt参数
from langchain.chat_models import  init_chat_model
from langchain.agents import  create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)
custom_profile={
    "max_input_tokens":128_000
}
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek",
                      profile=custom_profile)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]


agent=create_agent(model="deepseek-flash",
                   middleware=[
                       SummarizationMiddleware(
                           model=model,
                           trigger=[
                               ("tokens",100),
                               ("messages",6),
                               ("fraction",0.001)
                           ],
                           keep=("messages",2),
                           summary_prompt="对历史信息摘要，消息列表如下\n{messages}"
                       )
                   ])

response=agent.invoke({"messages":messages})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

历史信息摘要：  
系统设定AI为“非常友好的AI助手”。用户自称“老王”，询问AI身份；AI回答称自己是“小王”。随后老王表示很高兴认识小王。双方完成了友好问候和自我介绍。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

老王别误会，我刚才就是皮了一下，开了个没头没脑的玩笑，真没别的意思。能认识您我挺高兴的；要是那句话让您不舒服，我收回。咱重新来：很高兴认识您，老王！


### HumanInTheLoopMiddleware中间件

HumanInTheLoopMiddleware（人在环中间件、人工审核中间件）在工具调用前中断Agent运行，等待用户对工具调用请求决策。可选的决策有： approve（同意执行） 、 edit（编辑调用配置后执行） 、 reject（拒绝执行） 。

参数：interrupt_on 工具名和中断策略的映射

```
interrupt_on={
    "get_weather": True,
    "read_email_tool": False,
    "send_email_tool": {
        "allowed_decisions": ["approve", "reject"],
    },
}
```
1. True相当于所有决策都可以选择
2. False表示不中断，即无需审批即可执行
3. InterruptOnConfig 是一个TypedDict的子类，可以用字典直接赋值。支持的Key有：
   1. allowed_decisions 精细控制中断后允许的决策。
    2. description ：特定工具的中断描述信息，优先级高于description_prefix，后者会更改所有工具中断的描述。


参数2：description_prefix —自定义中断描述

In [5]:
# 调用前中断

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint
@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气
    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"
@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"
@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

agent=create_agent(model=model,
                   tools=[get_weather, get_news, read_email_tool,send_email_tool],
                   checkpointer=InMemorySaver(),
                   middleware=[
                       HumanInTheLoopMiddleware(
                            interrupt_on={
                                "get_weather":True,
                                "get_news":True,
                                "read_email_tool":False,
                                "send_email_tool": {
                                    "allowed_decisions": ["approve", "reject"],
                                    "description": "发送邮件中断啦"
                                },
                            },
                            description_prefix="中断啦"
                       ),
                   ],
                )

config = {"configurable": {"thread_id": "1"}}


response = agent.invoke(
    {
    "messages": [
        HumanMessage(content="请帮我查询今天北京的天气"
                            "查询今日新闻"
                            "查看ID为  'sk2131421' 的邮件内容，"
                            "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                            "同时做这四件事")
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()
# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)
# print("==== 逐个打印 interrupt 请求 ====")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为  'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='5ff4c50a-db79-4727-bcef-de8db6b8bfae'
        ),
        AIMessage(
            content='我来同时帮您完成这四件事。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'The user wants four independent things done. I should call all four tools in 
parallel.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 181,
                    'prompt_tokens': 532,
                    'total_tokens': 713,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 17,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 532
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '2aafb975-9842-4426-96ec-e238bcf54b94',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e244-ab07-72f8-9d83-196e950c5ab1-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_lUDNFrlhKb9uzQZpzvqn0252',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_D8Oz1Qb4N7txBlslhYC39297', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_sSlGf3PwHBaewU3VUOuo5472',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_Mk98CFqyCyfEp5uuvOwG0794',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 532,
                'output_tokens': 181,
                'total_tokens': 713,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 17}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断啦'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
                ]
            },
            id='8e3f0680220c1edf9e32876a7fbd8e21'
        )
    ]
}

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为  'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时帮您完成这四件事。
Tool Calls:
  get_weather (call_00_lUDNFrlhKb9uzQZpzvqn0252)
 Call ID: call_00_lUDNFrlhKb9uzQZpzvqn0252
  Args:
    city: 北京
  get_news (call_01_D8Oz1Qb4N7txBlslhYC39297)
 Call ID: call_01_D8Oz1Qb4N7txBlslhYC39297
  Args:
  read_email_tool (call_02_sSlGf3PwHBaewU3VUOuo5472)
 Call ID: call_02_sSlGf3PwHBaewU3VUOuo5472
  Args:
    email_id: sk2131421
  send_email_tool (call_03_Mk98CFqyCyfEp5uuvOwG0794)
 Call ID: call_03_Mk98CFqyCyfEp5uuvOwG0794
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
========== interrupts ==========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京'}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='8e3f0680220c1edf9e32876a7fbd8e21'
    )
]

{
    'name': 'get_weather',
    'args': {'city': '北京'},
    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京'}"
}

{'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
    'description': '发送邮件中断啦'
}

In [8]:
#指明工具调用请求决策
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "中国北京市", "is_forcast": True}
    }
}
news_decision = {
    "type": "approve",
}
send_email_decision = {
    "type": "approve"
}

decisions = {
    "decisions": []
}
interrupts=response.get("__interrupt__",[])
action_requests=interrupts[0].value["action_requests"]

for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    resumed_response=agent.invoke(
        Command(resume=decisions),
        config=config
    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为  'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时帮您完成这四件事。
Tool Calls:
  get_weather (call_00_lUDNFrlhKb9uzQZpzvqn0252)
 Call ID: call_00_lUDNFrlhKb9uzQZpzvqn0252
  Args:
    city: 中国北京市
    is_forcast: True
  get_news (call_01_D8Oz1Qb4N7txBlslhYC39297)
 Call ID: call_01_D8Oz1Qb4N7txBlslhYC39297
  Args:
  read_email_tool (call_02_sSlGf3PwHBaewU3VUOuo5472)
 Call ID: call_02_sSlGf3PwHBaewU3VUOuo5472
  Args:
    email_id: sk2131421
  send_email_tool (call_03_Mk98CFqyCyfEp5uuvOwG0794)
 Call ID: call_03_Mk98CFqyCyfEp5uuvOwG0794
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

中国北京市今天天气不错
明天下雨
================================= Tool Message ======================

### PIIMiddleware中间件

敏感信息保护
PII中间件用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略。

- 参数1：pii_type —检测的PII数据类型，可以是内置类型或自定义类型，自定义类型有
1. email ：电子邮箱地址
2. credit_card ：信用卡号
3. url ：网址
4. mac_address ：设备MAC地址
5. ip ：IP地址

- 参数2：strategy —处理PII信息的策略
支持四种选项：
1. redact ：将检测到的PII信息用字符串 [REDACTED_[PII_TYPE]] 替换，其中的 PII_TYPE是上面提到的具体类型，比如 [REDACTED_EMAIL] 、 [REDACTED_CREDIT_CARD] 这样的标签。完全 “擦除/隐藏” 真实内容。适合日志清洗、合规需求、公开输出时隐藏敏感内容。
2. mask ：用 *** 将PII信息的前面一部分信息遮蔽。比如信用卡号可能变成 ****-****-****-1234 （只保留最后几位/部分可见），邮箱可能保留域名部分 + 隐藏用户名的一部分等 — 既隐藏大部分敏感信息，又保留了一点“可辨识性”（比如账号后四位、域名等），适合用户服务界面/ 前端显示 / 需要部分可识别但不泄露完整敏感内容的场景。
3. hash ：用检测到的PII信息的 哈希值 替代原值。比如 <email_hash:a1b2c3d4> 。适合analytics、调试 (debug)、统计分析、匿名追踪等场景。
4. block ：如果检测到PII信息， 直接抛出异常 。适合对隐私要求极高、绝不允许泄露任何敏感信息的场景。


- 参数3：detector —自定义 PII检测函数 或者 正则表达式
如果没有提供则使用内置的检测函数。
LangChain为每种PII信息定制了专门的检测函数，相关源码如下
```
    BUILTIN_DETECTORS: dict[str, Detector] = {
        "email": detect_email,
        "credit_card": detect_credit_card,
        "ip": detect_ip,
        "mac_address": detect_mac_address,
        "url": detect_url,
    }
```

- 参数4：apply_to_input —是否在调用模型前检测
默认为True。
- 参数5：apply_to_output —是否在模型调用后检测
默认为False。
- 参数6：apply_to_tool_results —是否在工具调用后检测其输出
默认为False。
通常我们 只在模型调用前 检测。因为PII检测的主要目的是避免将敏感信息发送给模型服务导致信息泄露。

In [11]:
from langchain.chat_models import init_chat_model
from dotenv import  load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.agents.middleware import PIIMiddleware
import os
load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

agent=create_agent(model=model,
                   tools=[],
                   middleware=[PIIMiddleware("email",strategy="redact",apply_to_input=True),
                               PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
                               PIIMiddleware("url",strategy="hash",apply_to_input=True),
                               PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
                               PIIMiddleware("ip",strategy="block",apply_to_input=True),])

response = agent.invoke({
    "messages": [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()


try:
    response1 = agent.invoke({
        "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到异常，{e}")


================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

我无法直接替你执行这些操作：

1. **发送邮件**：我不能实际发送邮件到 `[REDACTED_EMAIL]`。建议你在自己的邮箱客户端中撰写并发送。
2. **查询银行卡余额**：不能查询或处理银行卡余额等敏感金融信息。请通过银行官方 App、官网或客服热线查询，切勿向他人泄露完整卡号、验证码、密码。
3. **访问链接** `<url_hash:dd5fc2a9>`：我不能访问该外部链接。若来源不明，请谨慎点击，避免钓鱼风险。
4. **确认 MAC 地址**：`**-**-**-**-**-11` 的形式符合 MAC 地址的常见格式：6 组、每组 2 位十六进制字符、以连字符分隔。但由于大部分字符被星号遮蔽，只能说是“格式上像 MAC 地址”，无法确认其完整有效性或归属设备。
检测到异常，Detected 1 instance(s) of ip in text content


###  ModelCallLimitMiddleware中间件

限制模型调用次数，避免无限循环，控制调用成本

In [2]:
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.messages import HumanMessage,AIMessage,SystemMessage,ToolMessage
from langchain.agents.middleware import ModelCallLimitMiddleware
from dotenv import  load_dotenv

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")


from typing import List
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, # 每个线程最多2次模型调用
            # run_limit=5, # 每次运行最多5次
            exit_behavior="end", # 达到限制后退出
        ),
    ],
)

def pretty_iterate_msg(messages:List[HumanMessage|SystemMessage|ToolMessage|AIMessage]):
    for msg in messages:
        msg.pretty_print()


config = {"configurable": {"thread_id": "1"}}
response_first = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])
response_second = agent.invoke({
    "messages": [HumanMessage("你是谁？")]},
    config=config
)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])
response_third = agent.invoke({
    "messages": [HumanMessage("你能帮我做什么？")]},
    config=config
)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你 😊

我是 DeepSeek，可以帮你解答问题、写作、翻译、编程、分析资料等等。有什么想聊的或需要帮忙的吗？
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你 😊

我是 DeepSeek，可以帮你解答问题、写作、翻译、编程、分析资料等等。有什么想聊的或需要帮忙的吗？
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 **DeepSeek**，一个由深度求索公司创造的 AI 助手 🤖

我擅长回答问题、写作、翻译、编程、资料分析等，也可以接收你上传的图片、PDF、Word、Excel 等文件并读取其中的文字信息来处理。

目前我是**免费**使用的，支持联网搜索（需要手动开启），上下文长度最高可达 **1M**，可以一次性处理比较大的资料。

有什么需要我帮忙的，尽管说 😊
============================== > thir

In [3]:
# 整个对话限制 抛出异常

from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.messages import HumanMessage,AIMessage,SystemMessage,ToolMessage
from langchain.agents.middleware import ModelCallLimitMiddleware
from dotenv import  load_dotenv

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")


from typing import List
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, # 每个线程最多2次模型调用
            # run_limit=5, # 每次运行最多5次
            exit_behavior="error",
        ),
    ],
)

def pretty_iterate_msg(messages:List[HumanMessage|SystemMessage|ToolMessage|AIMessage]):
    for msg in messages:
        msg.pretty_print()


config = {"configurable": {"thread_id": "1"}}
response_first = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])
response_second = agent.invoke({
    "messages": [HumanMessage("你是谁？")]},
    config=config
)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])
response_third = agent.invoke({
    "messages": [HumanMessage("你能帮我做什么？")]},
    config=config
)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])

============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你 😊

有什么我可以帮忙的吗？无论是聊天、解答问题、写东西、分析资料，还是别的什么，都可以告诉我。
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你 😊

有什么我可以帮忙的吗？无论是聊天、解答问题、写东西、分析资料，还是别的什么，都可以告诉我。
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

我是 AI 助手，可以理解你的问题并帮你解答、写作、分析、翻译、聊天等。

我是由深度求索（DeepSeek）开发的模型。简单说，我不是真人，而是一个人工智能助手 😊 有什么想聊或想让我帮忙的，直接说就行。


ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

### ToolCallLimitMiddleware中间件

限制工具调用次数，可以限制所有工具的调用次数，也可以限制特定工具的调用次数

退出行为有三种模式：
- error：直接抛异常
- end：结束整个会话
- continue：继续运行Agent，这是默认行为，此时Agent会将工具调用超出限制的信息传递给模型，后者自主决定后续行为，如果模型能力不足，可能导致死循环，为了避免这种情况，我们实现的fake server会以20%的概率输出正确响应，从而能终止循环。

In [9]:
# 整个会话限制

from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage
from langchain.chat_models import  init_chat_model
from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(model="deepseek-flash",
                        model_provider="deepseek",
                        reasoning_effort="none")
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), # Required for thread limiting
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2, # 每个线程最多2次工具调用
            run_limit=2, # 每次运行最多2次
            exit_behavior="end",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo],
    system_prompt=(
        "你是助手。只有当用户消息中明确包含姓名/邮箱/手机号等信息时，"
        "才调用 ContactInfo 工具；只有明确包含事件名称和日期时，才调用 EventInfo。"
        "其他情况（如打招呼、闲聊）直接用自然语言回复，不要调用任何工具。"
    )
)
def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage |AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)
pretty_iterate_msg(response["messages"])

================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_SFoD6aKJGMpxgy5Xc3gw6480)
 Call ID: call_00_SFoD6aKJGMpxgy5Xc3gw6480
  Args:
    name: 张伟
    email: zhangwei@example.com
    phone: 13800138000
  EventInfo (call_01_KicByQvzNIYoC73WIMLx1784)
 Call ID: call_01_KicByQvzNIYoC73WIMLx1784
  Args:
    event_name: 张伟的生日派对
    date: 2025年12月25日
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
==========================

In [15]:
# 整个会话限制

from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage
from langchain.chat_models import  init_chat_model
from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(model="deepseek-flash",
                        model_provider="deepseek",
                        reasoning_effort="none")
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), # Required for thread limiting
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2, # 每个线程最多2次工具调用
            run_limit=2, # 每次运行最多2次
            exit_behavior="error",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo],
    system_prompt=(
        "你是助手。只有当用户消息中明确包含姓名/邮箱/手机号等信息时，"
        "才调用 ContactInfo 工具；只有明确包含事件名称和日期时，才调用 EventInfo。"
        "其他情况（如打招呼、闲聊）直接用自然语言回复，不要调用任何工具。"
    )
)
def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage |AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)
pretty_iterate_msg(response["messages"])

ToolCallLimitExceededError: Tool call limit reached: run limit exceeded (3/2 calls).

In [17]:
# 整个会话限制

from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage
from langchain.chat_models import  init_chat_model
from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(model="deepseek-flash",
                        model_provider="deepseek",
                        reasoning_effort="none")
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(), # Required for thread limiting
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2, # 每个线程最多2次工具调用
            run_limit=2, # 每次运行最多2次
            exit_behavior="continue",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo],
    system_prompt=(
        "你是助手。只有当用户消息中明确包含姓名/邮箱/手机号等信息时，"
        "才调用 ContactInfo 工具；只有明确包含事件名称和日期时，才调用 EventInfo。"
        "其他情况（如打招呼、闲聊）直接用自然语言回复，不要调用任何工具。"
    )
)
def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage |AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)
pretty_iterate_msg(response["messages"])

================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_00_Ad9wynYGCVNodbAbWxCL9784)
 Call ID: call_00_Ad9wynYGCVNodbAbWxCL9784
  Args:
    name: 张三
    email: zhangsan@example.com
    phone: 13800138000
  EventInfo (call_01_PdMpsvj78ttixAubyiNk7348)
 Call ID: call_01_PdMpsvj78ttixAubyiNk7348
  Args:
    event_name: 产品发布会
    date: 2025-06-15
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
=============================

### LLMToolSelectorMiddleware中间件

当工具太多时，用子模型筛选最相关的几个工具

参数：
- model:用于筛选工具的子模型
- max_tools:限定可用的工具总数
- always_include:指定的工具不被计数

### ToolRetryMiddleware中间件

基于指数退避算法，设置工具调用失败时的重试策略。
jitter是为了避免大量工具的重试请求集中在固定的时间点，引入抖动。


In [9]:
 # 带抖动
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import datetime

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek",)

def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")

count = 1
start_time = None

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔{interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6, # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 =7 次）
            backoff_factor=2.0, # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=True, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,), # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
            ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_2KBeS4qJZQvj3dIJIRo00698)
 Call ID: call_00_2KBeS4qJZQvj3dIJIRo00698
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_L58vhNI4pMBiq9y7LS2Q5287)
 Call ID: call_00_L58vhNI4pMBiq9y7LS2Q5287
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

抱歉，天气查询工具目前不可用 / 未实现

In [10]:
 # 无抖动
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import datetime

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek",)

def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")

count = 1
start_time = None

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔{interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware 用于捕获工具执行中的异常并自动重试
        ToolRetryMiddleware(
            max_retries=6, # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 =7 次）
            backoff_factor=2.0, # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=False, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,), # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
            ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今天北京天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_zsJQ3XizsP00n1x17V1T5922)
 Call ID: call_00_zsJQ3XizsP00n1x17V1T5922
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

抱歉，我尝试查询北京天气时，天气服务接口返回了错误（超时/未实现），所以暂时拿不到实时数据。

建议你：
- 稍后再问我一次，我重试查询；
- 或者直接查看中国天气网、手机自带天气 App 等来源。

需要我现在再试一次吗？


![image.png](images/jitter.png)

在高并发的生产环境中，关闭jitter会引发“惊群效应”

- 没有 Jitter 的惨剧（ jitter=False ）：
假设某刻天气 API 服务突然宕机了 1 秒。此时刚好有 1000 个用户同时发起了查询。因为这 1000个请求同时失败，并且它们都严格死板地等待 1 秒、2 秒、4 秒……
这意味着，在第 1 秒、第 3 秒、第 7 秒的那个精准的时间点上，这 1000 个请求会整整齐齐地再次同时轰炸服务器。刚刚复活的服务器瞬间又被这波整齐的峰值流量压垮，形成恶性循环。

- 引入 Jitter 的优势（ jitter=True ）：
通过给重试时间加上随机性，这 1000 个请求会在 秒、 秒的区间内均匀地错开（削峰填谷）。流量被平摊到了整条时间轴上，服务器就能轻松地分批处理完这些请求。

### ModelRetrymiddleware

模型调用失败时重试，策略和工具调用的重试一样，都是基于指数退避算法

In [13]:
# continue继续运行
from langchain.agents import  create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import  load_dotenv

load_dotenv(override=True)

agent=create_agent(model="deepseek-fk",
                   middleware=[
                        ModelRetryMiddleware(
                            max_retries=6,
                            backoff_factor=2.0,
                            initial_delay=1.0,
                            max_delay=10.0,
                            on_failure="continue",
                            jitter=False
                        )
                   ])

response=agent.invoke({
    "messages":[HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

Model call failed after 7 attempts with BadRequestError: Error code: 400 - {'error': {'message': 'The supported API model names are deepseek-flash, deepseek-v4-pro, but you passed deepseek-fk. (request_id: 4c2eec6a-70cc-4b2f-a9ef-b0e8f2693d83)', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


In [14]:
# error抛出异常
from langchain.agents import  create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import  load_dotenv

load_dotenv(override=True)

agent=create_agent(model="deepseek-fk",
                   middleware=[
                        ModelRetryMiddleware(
                            max_retries=6,
                            backoff_factor=2.0,
                            initial_delay=1.0,
                            max_delay=10.0,
                            on_failure="error",
                            jitter=False
                        )
                   ])

response=agent.invoke({
    "messages":[HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

BadRequestError: Error code: 400 - {'error': {'message': 'The supported API model names are deepseek-flash, deepseek-v4-pro, but you passed deepseek-fk. (request_id: 9fbddaa2-3a9d-493c-8b24-516528d09087)', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

### ContextEditingMiddleware中间件

上下文编辑中间件，该中间件提供了上下文管理的一种方式。 通过更改发送给模型的消息列表来控制成本。
注意：不会更改消息列表。因此我们只能通过token用量来推测是否对消息列表进行了裁剪。

In [15]:
    from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware,ClearToolUsesEdit
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv(override=True)
count = 0
@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
    f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
    f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
    f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
    f"真不错，天气非常好，适合出行，这里token挺多的"
    f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model="deepseek-chat",
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50,
                    keep=0,
                    ),
                ],
        ),
    ],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}
for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]},
    config=config
    )
print("---- 本次返回的 messages ----")
for msg in response["messages"]:
    # msg.pretty_print()
    if isinstance(msg, AIMessage):
        if not msg.tool_calls:
            print(f"本次token用量：{msg.usage_metadata}")

============================== 当前是第 1 轮调用 ==============================
============================== 当前是第 2 轮调用 ==============================
============================== 当前是第 3 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 330, 'output_tokens': 25, 'total_tokens': 355, 'input_token_details': {'cache_read': 128}, 'output_token_details': {}}
本次token用量：{'input_tokens': 426, 'output_tokens': 29, 'total_tokens': 455, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
本次token用量：{'input_tokens': 526, 'output_tokens': 29, 'total_tokens': 555, 'input_token_details': {'cache_read': 384}, 'output_token_details': {}}


. ContextEditingMiddleware 的价值：大模型多轮对话时，如果频繁调用产生大量文本的工具（如代码执行、网页爬取），历史记录会急剧膨胀。这个中间件就像一个“上下文抽脂手术”，在不影响当前对话的前提下，自动在后台删掉之前沉淀的工具调用废话，从而极大地节省 Token 费用并防止超出模型最大上下文窗口（Context Window）。

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware,ClearToolUsesEdit
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv(override=True)
count = 0
@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
    f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
    f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
    f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
    f"真不错，天气非常好，适合出行，这里token挺多的"
    f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model=model,
    tools=[get_weather],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}
for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]},
    config=config
    )
print("---- 本次返回的 messages ----")
for msg in response["messages"]:
    # msg.pretty_print()
    if isinstance(msg, AIMessage):
        if not msg.tool_calls:
            print(f"本次token用量：{msg.usage_metadata}")

============================== 当前是第 1 轮调用 ==============================
============================== 当前是第 2 轮调用 ==============================
============================== 当前是第 3 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 482, 'output_tokens': 30, 'total_tokens': 512, 'input_token_details': {'cache_read': 256}, 'output_token_details': {'reasoning': 13}}
本次token用量：{'input_tokens': 687, 'output_tokens': 17, 'total_tokens': 704, 'input_token_details': {'cache_read': 512}, 'output_token_details': {'reasoning': 0}}
本次token用量：{'input_tokens': 651, 'output_tokens': 36, 'total_tokens': 687, 'input_token_details': {'cache_read': 384}, 'output_token_details': {'reasoning': 19}}


## 自定义中间件

某些复杂场景下，官方内置的中间件不能完全满足需求，此时可以通过实现LangChain暴露的中间件hook函数 构建自定义中间件。

![image.png](images/hook.png)

In [18]:
# 基于装饰器实现

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents.middleware import before_model, after_model,before_agent, after_agent, AgentState, AgentMiddleware
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)

model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model <- "
    return None

@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model <- "
    return None

@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent <- "
    return None

@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    state["messages"][-1].content += " -> after_agent <- "
    return None

agent=create_agent(model=model,
                   middleware=[before_model_middleware,after_model_middleware,before_agent_middleware,after_agent_middleware])

response=agent.invoke({
    "messages":[HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？

（注意到你的消息里带了 `before_agent`、`before_model` 这样的标记，像是某个 Agent 框架的回调/钩子名称。如果你是在调试这类框架，也可以直接说需求，比如排查回调没触发、顺序问题等。） -> after_model <-  -> after_agent <-


### 基于类实现

关键规则：
1. 必须继承 AgentMiddleware ← 这个固定
2. 方法名固定 ( before_model , after_model ) ← 这个固定
3. 类名随意 ← 这个不固定

In [19]:
from langchain.agents.middleware import AgentMiddleware, AgentState,hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any

class MyMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        state["messages"][-1].content += " -> before_model <- "
        return None
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        state["messages"][-1].content += " -> after_model <- "
        return None
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        state["messages"][-1].content += " -> before_agent <- "
        return None
    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        state["messages"][-1].content += " -> after_agent <- "
        return None

my_middleware = MyMiddleware()
agent = create_agent(
    model = model,
    middleware = [my_middleware]
)
response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！👋

我看到你提到了 `before_agent` 和 `before_model`，它们看起来像是某些 Agent/模型框架里的钩子（hook）或生命周期节点。你想了解它们的区别、执行顺序，还是想让我帮你排查相关配置？可以把具体问题或代码贴出来，我帮你看看。 -> after_model <-  -> after_agent <-


### 参数说明

Node-style hooks函数有两个参数
- state: 是一个AgentState实例，维护Agent运行过程中的状态，这类状态会随着Agent的运行而发生变化，包括 消息列表 。
- runtime: 是一个Runtime实例，维护Agent运行过程中的上下文环境，包括 上下文 、 长期记忆 等。

### 装饰器参数 can_jump_to

钩子函数可以 改变Agent正常的运行轨迹 。比如：发现上下文窗口溢出，直接跳转至结尾，提前终止整个Agent。

can_jump_to 决定了钩子函数可以直接跳转至流程的哪些位置，可取值如下：

- end：跳转至Agent流程末尾，或第一个after_agent钩子，直接终止整个流程。
- tools：跳转至工具节点。
- model：跳转至模型节点，或第一个before_model钩子

In [6]:
# 基于装饰器实现
from typing import Any
from dotenv import  load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, after_model,AgentState
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime

load_dotenv(override=True)
model =init_chat_model(model="deepseek-flash",
                       model_provider="deepseek",
                       max_retries=2,
                       timeout=30)

@tool
def get_news() -> str:
    """获取当日新闻"""
    return f"美加墨世界杯今日开幕"

# 在模型（LLM）执行前触发。允许跳转到 "tools" 节点。
@before_model(can_jump_to=["tools"])
def force_tool_first(state: AgentState, runtime: Runtime) -> dict[str, Any]| None:
    """
    【业务场景：强行拦截并触发工具】
    如果用户输入包含 "direct tool"，则跳过本次大模型的思考/生成阶段，
    直接伪造一个大模型的 tool_calls 意图，强行把控制权移交给工具执行节点。
    """
    text = state["messages"][-1].content
    # 检查关键词，满足条件则强行干预流程
    if isinstance(text, str) and "direct tool" in text.lower():
        print("[MIDDLEWARE] before_model: jump_to='tools'")
    # 人工构造一个大模型的消息对象（AIMessage）
    # 欺骗系统，让系统误以为这是模型自己决定要调用的工具
    fake_tool_call = AIMessage(
        content="人工构造的消息",
        tool_calls=[
            {
                "name": "get_news",
                "args": {},
                "id": "call_force_weather_001",
            }
        ],
    )
    # 返回更新后的状态：注入伪造的消息，并明确指定下一步跳转到 "tools" 节点
    return {
        "messages": [fake_tool_call],
        "jump_to": "tools",
    }
    # 如果不满足触发条件，返回 None，流程正常向下流转（继续让 LLM 思考）
    return None

# 在模型（LLM）执行生成之后触发。允许重新跳转回 "model" 节点。
@after_model(can_jump_to=["model"])
def retry_with_extra_instruction(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    """
    【业务场景：反思/重试机制】
    如果大模型已经生成了回答，但发现用户最初的请求包含 "retry model"，
    则动态追加一条系统提示词（SystemMessage），强行让模型重新生成（重试）一次。
    """
    # 倒序遍历消息历史，找到最近的一次用户输入（human 消息）
    user_text = ""
    for msg in reversed(state["messages"]):
        if getattr(msg, "type", "") == "human":
            user_text = getattr(msg, "content", "")
            break
    # 检查用户输入是否包含触发重试的关键字
    if isinstance(user_text, str) and "retry model" in user_text.lower():
        # 【核心防御】：防止无限循环重跳（死循环）
        # 检查消息历史中是否已经注入过这条特殊的系统提示。如果有，说明已经重试过了，不再重复干预。
        already_injected = any(
            isinstance(getattr(msg, "content", None), str)
            and "你必须以【二次回答】开头" in msg.content
            for msg in state["messages"]
        )
    if already_injected:
        return None # 已注入过，直接放行，结束重试流程
    print("[MIDDLEWARE] after_model: jump_to='model' with extra systeminstruction")
    # 返回更新后的状态：追加强力约束的系统消息，并将指针跳回 "model" 节点重新执行
    return {
        "messages": [
            SystemMessage("你必须以【二次回答】开头，并且只用一句话回答。")
        ],
        "jump_to": "model",
    }
    return None


# 在模型（LLM）执行前触发。允许直接跳转到 "end" 节点（强行终止）。
@before_model(can_jump_to=["end"])
def overflow_context_processor(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    """
    【业务场景：安全卫士/异常拦截】
    模拟上下文窗口溢出（Token超限）或其他严重的系统阻断情况。
    一旦触发，直接熔断流程，拒绝让大模型继续处理，直接报错或返回兜底文案。
    """
    # 假装溢出,模拟检查最后一条消息是否包含 overflow 标识
    if "overflow" in state["messages"][-1].content:
        print("[MIDDLEWARE] before_model: jump_to='end' when contenxtwindow overflow")
    # 构造兜底的结束消息，并直接指定跳转到 "end" 终止 Agent 运行
    return {
        "messages": [
            AIMessage("上下文窗口溢出，终止")
        ],
        "jump_to": "end",
    }

agent=create_agent(model=model,
                   tools=[get_news],
                   middleware=[force_tool_first,retry_with_extra_instruction,overflow_context_processor])

def run_once(user_input: str):
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": user_input}
            ]
        }
    )
    for msg in result["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    # Case 1: 直接跳 tools
    # 预期表现：
    # 1. 触发 force_tool_first，打印 "[MIDDLEWARE] before_model:jump_to='tools'"
    # 2. 绕过 LLM 的首轮思考，直接调用 `get_news` 工具
    # 3. 工具返回结果后，LLM 总结工具结果并输出
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("请帮我查今日新闻 direct tool")
    # Case 2: 输出后跳回 model
    # 预期表现：
    # 1. 正常进入 LLM 生成第 1 版回答
    # 2. 触发 retry_with_extra_instruction，打印 "[MIDDLEWARE] after_model:jump_to='model'..."
    # 3. 注入系统提示词后，LLM 被强行拉回并生成第 2 版回答
    # 4. 最终输出应带有“【二次回答】”前缀
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("请随便介绍一下 LangChain retry model")
    # Case 3:
    # 预期表现：
    # 1. 触发 overflow_context_processor 中间件
    # 2. 直接打印终止信息并退出，LLM 根本不会接收到这个请求
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("你好 overflow")
    # Case 4: 正常流程
    # 预期表现：
    # 1. 没有任何中间件被触发（不满足任何关键字）
    # 2. Agent 走正常的 OOTB（Out of the box）标准工作流：User -> Model -> CallTool -> Model -> End
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日新闻摘要？")

============================== -> Case 1 <- ==============================
[MIDDLEWARE] before_model: jump_to='tools'


KeyboardInterrupt: 

### Wrap-style hooks函数用法

我们可以同时在模型调用前后做事，所以命名为 wrap_model_call ，wrap意为包裹

源码：
```
    def wrap_model_call(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
        """
        request: 包含 model, messages, system_message, tools, state
        handler: 执行实际模型调用的函数
        返回：ModelResponse
        """
```


In [7]:
#基于装饰器实现
from langchain.agents.middleware import wrap_model_call, ModelRequest,ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable
from langchain.chat_models import  init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

@wrap_model_call
def wrap_model_call_middleware(
    request: ModelRequest, # 包含即将发送给大模型的所有请求数据（如消息列表、温度等）
    handler: Callable[[ModelRequest], ModelResponse], # 核心句柄：代表下一个中间件或最终的大模型调用服务
) -> ModelResponse | None:
    # 动态篡改用户发出的最后一条消息的内容，悄悄往里面追加字符串。
    # 典型应用：统一在底层为所有请求追加特殊的 Prompt 提示词（例如：“请用中文回答”、“禁止透漏公司机密”等）。
    request.messages[-1].content += " -> wrap_model_call_before <- "
    # 将修改后的请求传递给 handler，真正去调用大模型（或者流转到下一个拦截器）
    # 这一步会产生真实的 Token 消耗并等待大模型响应
    response = handler(request)
    # 大模型返回响应后，在将响应交付给 Agent 状态机之前，对其内容进行直接篡改
    # `response.result` 是一个消息列表，修改其第一条返回消息的内容
    # 典型应用：做底层的文本敏感词过滤、输出格式强行格式化、或是统一添加某些后处理标记。
    response.result[0].content += " -> wrap_model_call_after <- "
    # 将修改完的响应体返回，继续维持 Agent 生命周期流转
    return response

agent = create_agent(
    model = model,
    middleware = [wrap_model_call_middleware]
)
response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> wrap_model_call_before <- 
================================== Ai Message ==================================

你好！👋

我看到你提到了 `wrap_model_call_before` —— 这看起来像是某个框架/代码里的钩子（hook）名称，但我这边没有关于它的上下文信息。

你想做什么呢？比如：
- 想了解某个库中 `wrap_model_call_before` 这类钩子的用法？
- 遇到了报错或行为异常，需要排查？
- 还是只是打个招呼、测试一下？

告诉我更多细节（用的什么框架、想实现什么），我好帮你。 -> wrap_model_call_after <-


In [8]:
# 基于类实现

from langchain.agents.middleware import AgentMiddleware, ModelRequest,ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable
class WrapModelCallMiddleWare(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse | None:

        request.messages[-1].content += " -> wrap_model_call_before <- "
        response = handler(request)
        response.result[0].content += " -> wrap_model_call_after <- "
        return response

agent = create_agent(
    model = model,
    middleware = [WrapModelCallMiddleWare()]
)
response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> wrap_model_call_before <- 
================================== Ai Message ==================================

你好！👋

看到你写了 `-> wrap_model_call_before <-`，这看起来像是某个**模型调用前置包装钩子（before-hook / middleware）**的命名。想先确认一下你的意图：

- 是想了解这类钩子**一般干什么**？
- 还是在某个框架里（LangChain / LangGraph、自研 Agent 框架、FastAPI 中间件等）遇到了**具体问题**？
- 或者只是想让我**实现**一个叫 `wrap_model_call_before` 的函数？

简单说，这种 `before` 包装器通常长这样：

```python
def wrap_model_call_before(fn):
    def wrapper(messages, **kwargs):
        # 1. 调用前拦截：改 messages、注入系统提示、加缓存/日志/限流
        # 2. 调底层模型
        result = fn(messages, **kwargs)
        # 3. 可选：后处理
        return result
    return wrapper
```

典型用途：注入上下文、参数改写（temperature/model）、token 计数、重试、审计日志、成本统计、敏感词过滤。

你可以补充一下：**框架/语言**、**想达到的效果**，以及 `-> ... <-` 这个箭头是不是表示“在这里插入钩子”。我再给你更贴合的方案。 -> wrap_model_call_after <-


#### wrap_tool_call

我们可以同时在工具调用前后做事，所以命名为 wrap_tool_cal


In [9]:
# 基于装饰器实现

from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.types import Command
from typing import Callable
@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气
    Args:
    city: 城市名称
    is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res

@wrap_tool_call
def wrap_tool_call_middleware(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果： {result}")
    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)
    print(f"更新后的参数：{request.tool_call['args']}")
    print(f"更新参数调用结果： {result}")
    return result

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)
response = agent.invoke({
    "messages": [HumanMessage("你好啊，今天杭州的天气怎么样")]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '杭州', 'is_forcast': False}
原始参数调用结果： content='杭州今天天气不错' name='get_weather' tool_call_id='call_00_UEKreeDzUR6u4h3En2R42645'
更新后的参数：{'city': '杭州', 'is_forcast': True}
更新参数调用结果： content='杭州今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_UEKreeDzUR6u4h3En2R42645'
================================ Human Message =================================

你好啊，今天杭州的天气怎么样
================================== Ai Message ==================================

你好！我帮你查一下杭州今天的天气～
Tool Calls:
  get_weather (call_00_UEKreeDzUR6u4h3En2R42645)
 Call ID: call_00_UEKreeDzUR6u4h3En2R42645
  Args:
    city: 杭州
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

杭州今天天气不错
明天天气也很好
================================== Ai Message ==================================

我查到的情况是这样的：

- **杭州今天**：天气不错 ☀️
- **明天**：天气也很好

不过这个查询结果比较笼统，没有给出具体的气温、湿度、风力等细节。如果你需要更精确的数据，比如具体温度和穿衣建议，可以看看手机上的天气 App，或者告诉我你更关心哪方面，我再帮你查一次～

需要的话，我也可以顺便帮你看看其他城市的天气。


In [10]:
# 基于类实现

from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.types import Command
from typing import Callable
@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气
    Args:
    city: 城市名称
    is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        result=handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果： {result}")
        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)
        print(f"更新后的参数：{request.tool_call['args']}")
        print(f"更新参数调用结果： {result}")
        return result


agent=create_agent(model=model,
                   tools=[get_weather],
                   middleware=[WrapToolCallMiddleware()])

response=agent.invoke({
    "messages":[HumanMessage("南京天气怎么样")]
})

for msg in response["messages"]:
    msg.pretty_print()


原始参数：{'city': '南京', 'is_forcast': False}
原始参数调用结果： content='南京今天天气不错' name='get_weather' tool_call_id='call_00_Ydn67PY3b466b150v3fi5852'
更新后的参数：{'city': '南京', 'is_forcast': True}
更新参数调用结果： content='南京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_Ydn67PY3b466b150v3fi5852'
================================ Human Message =================================

南京天气怎么样
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_Ydn67PY3b466b150v3fi5852)
 Call ID: call_00_Ydn67PY3b466b150v3fi5852
  Args:
    city: 南京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

南京今天天气不错
明天天气也很好
================================== Ai Message ==================================

南京今天天气不错，明天天气也很好 ☀️

如果你想了解具体的气温、是否需要带伞等细节，或者想查询其他城市，随时告诉我～
